# Diagnóstico de Plantas Domésticas

**Aluno:** Henrique de Andrade França  
**Matrícula:** 20210024961

Instalação e Preparação do Ambiente

In [ ]:
!pip install experta
!pip install --upgrade frozendict

from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, P, AND, OR, NOT

  Preparing metadata (setup.py) ... done
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=b54e4b21b8f7d5951a8acf909024e8bf28e64464009d7564897dc8dae8ddbb33
  Stored in directory: /root/.cache/pip/wheels/f6/ff/aa/750fec7bf9618d87b53572def5abf3e098f853cc5ab4147656
Successfully built frozendict
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.7
    Uninstalling frozendict-2.4.7:
      Successfully uninstalled frozendict-2.4.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into accou

Representação do Conhecimento (Fatos / Memória de Trabalho)

In [ ]:
# fatos de entrada (fornecidos pelo usuario)
class Planta(Fact):
    """Informações gerais da planta. Campos: ambiente (str), aspecto_geral (str)"""
    pass

class Sintoma(Fact):
    """Sintomas visíveis. Campos: solo (str), folhas (str)"""
    pass

# fatos intermediarios (gerados pelo motor de inferencia)
class Estado(Fact):
    """Estado hídrico ou de saúde inferido. Campos: condicao (str)"""
    pass

class Diagnostico(Fact):
    """Problema específico identificado. Campos: problema (str)"""
    pass

# fato de saida (conclusao)
class Recomendacao(Fact):
    """Ação corretiva a ser tomada. Campos: acao (str)"""
    pass

Motor de Inferência e Base de Regras

In [ ]:
class DoutorPlanta(KnowledgeEngine):

    # ---------------------------------------------------------------------
    # RESOLUCAO DE CONFLITO (salience)
    # R1: se a planta ja estiver completamente seca e morta, paramos
    # salience=100 garante que dispare antes de qualquer tentativa de diagnostico
    # ---------------------------------------------------------------------
    @Rule(Planta(aspecto_geral="morta"), salience=100)
    def r1_planta_morta(self):
        print("[TRACE - R1] A planta foi declarada morta. Interrompendo diagnóstico.")
        self.declare(Recomendacao(acao="Infelizmente, a planta não tem salvação. Descarte e tente novamente com uma nova muda."))

    # ---------------------------------------------------------------------
    # NIVEL DE ENCADEAMENTO 1: sintomas -> estado da planta
    # NOT(Recomendacao()) impede que essas regras rodem se a planta estiver morta (R1)
    # ---------------------------------------------------------------------
    @Rule(NOT(Recomendacao()), Sintoma(solo="seco", folhas="murchas"))
    def r2_falta_agua(self):
        print("[TRACE - R2] Solo está seco e folhas murchas.")
        self.declare(Estado(condicao="desidratada"))

    @Rule(NOT(Recomendacao()), Sintoma(solo="encharcado", folhas="amarelas"))
    def r3_excesso_agua(self):
        print("[TRACE - R3] Solo está encharcado e folhas amarelando.")
        self.declare(Estado(condicao="excesso_agua"))

    # OR para agrupar sintomas semelhantes
    @Rule(NOT(Recomendacao()),
          OR(Sintoma(folhas="furadas"), Sintoma(folhas="comidas")))
    def r4_presenca_pragas(self):
        print("[TRACE - R4] Folhas apresentam danos físicos (furos ou pedaços faltando).")
        self.declare(Estado(condicao="ataque_pragas"))

    @Rule(NOT(Recomendacao()), Sintoma(solo="umido", folhas="verdes"))
    def r5_planta_saudavel(self):
        print("[TRACE - R5] Solo úmido ideal e folhas verdes.")
        self.declare(Estado(condicao="saudavel"))


    # ---------------------------------------------------------------------
    # NIVEL DE ENCADEAMENTO 2: estado + ambiente -> diagnostico
    # O motor cruza o estado gerado no nivel 1 com o ambiente onde a planta fica
    # ---------------------------------------------------------------------
    @Rule(Estado(condicao="desidratada"), Planta(ambiente="sol_direto"))
    def r6_diag_queimadura(self):
        print("[TRACE - R6] Inferência: Desidratada + Sol Direto = Estresse térmico severo.")
        self.declare(Diagnostico(problema="queimadura_solar_seca"))

    @Rule(Estado(condicao="desidratada"), Planta(ambiente="sombra"))
    def r7_diag_esquecimento(self):
        print("[TRACE - R7] Inferência: Desidratada na sombra = Apenas esquecimento de rega.")
        self.declare(Diagnostico(problema="esquecimento_rega"))

    @Rule(Estado(condicao="excesso_agua"))
    def r8_diag_raiz_podre(self):
        print("[TRACE - R8] Inferência: Excesso de água prolongado causa podridão.")
        self.declare(Diagnostico(problema="apodrecimento_raiz"))

    @Rule(Estado(condicao="ataque_pragas"))
    def r9_diag_lagartas(self):
        print("[TRACE - R9] Inferência: Insetos mastigadores identificados.")
        self.declare(Diagnostico(problema="infestacao_lagartas"))


    # ---------------------------------------------------------------------
    # NIVEL DE ENCADEAMENTO 3: diagnostico -> recomendacao (acao final)
    # ---------------------------------------------------------------------
    @Rule(Diagnostico(problema="queimadura_solar_seca"))
    def r10_acao_queimadura(self):
        print("[TRACE - R10] Gerando recomendação para estresse térmico.")
        self.declare(Recomendacao(acao="Regar por imersão (balde) e mover para local de meia-sombra imediatamente."))

    @Rule(Diagnostico(problema="esquecimento_rega"))
    def r11_acao_rega_simples(self):
        print("[TRACE - R11] Gerando recomendação para rega atrasada.")
        self.declare(Recomendacao(acao="Fazer uma rega abundante. Criar um alarme semanal no celular."))

    @Rule(Diagnostico(problema="apodrecimento_raiz"))
    def r12_acao_replantio(self):
        print("[TRACE - R12] Gerando recomendação para replantio de emergência.")
        self.declare(Recomendacao(acao="Retirar a planta do vaso, cortar raízes escuras/moles e replantar em terra nova e seca."))

    @Rule(Diagnostico(problema="infestacao_lagartas"))
    def r13_acao_pesticida(self):
        print("[TRACE - R13] Gerando recomendação para controle de pragas.")
        self.declare(Recomendacao(acao="Remover lagartas manualmente e aplicar óleo de Neem ao final do dia."))

    # ---------------------------------------------------------------------
    # REGRA DE IMPRESSAO FINAL
    # ---------------------------------------------------------------------
    @Rule(Recomendacao(acao=MATCH.a))
    def r14_imprimir(self, a):
        print(f"\n=======================================================")
        print(f"PRESCRIÇÃO FINAL")
        print(f"O que você deve fazer: {a}")
        print(f"=======================================================\n")

1. **Nível 1:** A regra R2 olha para os sintomas físicos informados pelo usuário (solo seco e folhas murchas) e conclui que a planta está com falta de água, declarando o fato intermediário `Estado(condicao="desidratada")`.

2. **Nível 2:** A regra R7 só funciona porque junta o ambiente onde a planta fica (`sombra`) com a conclusão anterior do motor (`desidratada`). A partir desse cruzamento, ela gera um segundo fato intermediário: `Diagnostico(problema="esquecimento_rega")`.

3. **Nível 3:** A regra R11 utiliza estritamente essa nova informação gerada no Nível 2 (`esquecimento_rega`) para prescrever a intervenção adequada, alcançando a Recomendacao final (que é regar abundantemente e criar um alarme).

Execução e Casos de Teste

In [ ]:
print(">>> CASO DE TESTE 1: Encadeamento Profundo (Esquecimento na Sombra)")
# caminho esperado: R2 -> R7 -> R11 -> R14
engine = DoutorPlanta()
engine.reset()
engine.declare(Planta(aspecto_geral="viva", ambiente="sombra"))
engine.declare(Sintoma(solo="seco", folhas="murchas"))
engine.run()

print("\n" + "="*60 + "\n")

print(">>> CASO DE TESTE 2: Resolução de Conflitos (Planta Morta)")
# caminho esperado: R1 dispara por causa da Salience 100
# NOT() impede que o motor tente avaliar o solo ou folhas
engine.reset()
engine.declare(Planta(aspecto_geral="morta", ambiente="sol_direto"))
engine.declare(Sintoma(solo="seco", folhas="murchas"))
engine.run()

print("\n" + "="*60 + "\n")

print(">>> CASO DE TESTE 3: Excesso de Cuidado (Afogando a planta)")
# caminho esperado: R3 -> R8 -> R12 -> R14
engine.reset()
engine.declare(Planta(aspecto_geral="viva", ambiente="sombra"))
engine.declare(Sintoma(solo="encharcado", folhas="amarelas"))
engine.run()

>>> CASO DE TESTE 1: Encadeamento Profundo (Esquecimento na Sombra)
[TRACE - R2] Solo está seco e folhas murchas.
[TRACE - R7] Inferência: Desidratada na sombra = Apenas esquecimento de rega.
[TRACE - R11] Gerando recomendação para rega atrasada.

PRESCRIÇÃO FINAL
O que você deve fazer: Fazer uma rega abundante. Criar um alarme semanal no celular.



>>> CASO DE TESTE 2: Resolução de Conflitos (Planta Morta)
[TRACE - R1] A planta foi declarada morta. Interrompendo diagnóstico.

PRESCRIÇÃO FINAL
O que você deve fazer: Infelizmente, a planta não tem salvação. Descarte e tente novamente com uma nova muda.



>>> CASO DE TESTE 3: Excesso de Cuidado (Afogando a planta)
[TRACE - R3] Solo está encharcado e folhas amarelando.
[TRACE - R8] Inferência: Excesso de água prolongado causa podridão.
[TRACE - R12] Gerando recomendação para replantio de emergência.

PRESCRIÇÃO FINAL
O que você deve fazer: Retirar a planta do vaso, cortar raízes escuras/moles e replantar em terra nova e seca.

